# Energy Reconstruction Using CNN - Both Charges and Cos(Zenith)

In [1]:
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt

from datetime import datetime

from tensorflow import keras
from keras import layers
from keras import models
from keras import callbacks
from keras import backend

#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import Dense, Conv2D, Flatten
#from tensorflow.keras.callbacks import ModelCheckpoint

from data_tools import load_preprocessed, dataPrep, nameModel
from hex_filter import hex_filter, MaskedConv2D

simPrefix = os.getcwd()+'\\simdata'

In [2]:
import functools

In [2]:
tf.__version__

'2.6.2'

## Model Design

In [3]:
# Name for model
name = 'hex'
i = 0
while(os.path.exists('untrainedModels/{}.h5'.format(name+str(i)))):
    i = i + 1
name = name+str(i)

# Data preparation: no merging of charge (q), no time layers included (t=False), data normalized from 0-1
prep = {'q':None, 't':False, 'normed':True, 'reco':'plane', 'cosz':True}
print(name)

hex1


In [4]:
# Create model using functional API for multiple inputs
charge_input=keras.Input(shape=(10,10,2,))

#x = layers.Conv2D(filters=1, 
#                      kernel_size = 3,
#                      kernel_initializer=hex_filter,
#                      strides=2, 
#                      padding='valid') (charge_input)

#conv1_layer = layers.Conv2D(64,kernel_size=3,padding='same',activation='relu')(charge_input)
#conv1_layer = layers.Conv2D(filters=64,kernel_size=3,kernel_initializer=hex_filter,padding='same',activation='relu')(charge_input)
conv1_layer = MaskedConv2D(filters=64,kernel_size=3,padding='same',activation='relu')(charge_input)
#conv1_layer = MaskedConv2D(num_outputs=64,padding='same')(charge_input)
#activ1_layer = layers.Activation('relu')(conv1_layer)

#conv2_layer = layers.Conv2D(32,kernel_size=3,padding='same',activation='relu')(conv1_layer)
#conv2_layer = layers.Conv2D(32,kernel_size=3,kernel_initializer=hex_filter,padding='same',activation='relu')(conv1_layer)
conv2_layer = MaskedConv2D(32,kernel_size=3,padding='same',activation='relu')(conv1_layer)
#conv2_layer = MaskedConv2D(num_outputs=32,padding='same')(activ1_layer)
#activ2_layer = layers.Activation('relu')(conv2_layer)

#conv3_layer = layers.Conv2D(16, kernel_size=3, padding='same',activation="relu")(conv2_layer)
#conv3_layer = layers.Conv2D(16, kernel_size=3,kernel_initializer=hex_filter, padding='same',activation="relu")(conv2_layer)
conv3_layer = MaskedConv2D(16, kernel_size=3, padding='same',activation="relu")(conv2_layer)
#conv3_layer = MaskedConv2D(num_outputs=16,padding='same')(activ2_layer)
#activ3_layer = layers.Activation('relu')(conv3_layer)

flat_layer = layers.Flatten()(conv3_layer)
#flat_layer = layers.Flatten()(activ3_layer)

#flat_layer = layers.Flatten()(conv3_layer)
zenith_input=keras.Input(shape=(1,))
concat_layer = layers.Concatenate()([flat_layer,zenith_input])

dense1_layer = layers.Dense(256,activation='relu')(concat_layer)

dense2_layer = layers.Dense(256,activation='relu')(dense1_layer)

dense3_layer = layers.Dense(256,activation="relu")(dense2_layer)

output = layers.Dense(1)(dense3_layer)

model = models.Model(inputs=[charge_input,zenith_input],outputs=output,name=name)

model.compile(loss='mean_squared_error', optimizer='adam', metrics=['mae','mse'])

TypeError: 'NoneType' object is not subscriptable

In [5]:
model.summary()

Model: "hex1"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 10, 10, 2)]  0                                            
__________________________________________________________________________________________________
masked_conv2d (MaskedConv2D)    (None, 10, 10, 64)   64          input_1[0][0]                    
__________________________________________________________________________________________________
activation (Activation)         (None, 10, 10, 64)   0           masked_conv2d[0][0]              
__________________________________________________________________________________________________
masked_conv2d_1 (MaskedConv2D)  (None, 10, 10, 32)   32          activation[0][0]                 
_______________________________________________________________________________________________

In [63]:
model.save('untrainedModels/%s.h5'% name)
np.save('untrainedModels/%s.npy' % name,prep)

In [5]:
class MaskedConv2D(tf.keras.layers.Layer):
    def __init__(self, *args, **kwargs):
        super(MaskedConv2D, self).__init__()
        self.conv2d = tf.keras.layers.Conv2D(*args, **kwargs)
        
    def build(self, input_shape):
        self.conv2d.build(input_shape[0])
        self._convolution_op = self.conv2d._convolution_op
        
    def masked_convolution_op(self, filters, kernel, mask):
        return self._convolution_op(filters, tf.math.multiply(kernel, tf.reshape(mask, mask.shape + (1,1) )))
        
    def call(self, inputs):
        x, mask = inputs
        mask=tf.constant([[0,1,1],[1,1,1],[1,1,0]], dtype=tf.float32)
        self.conv2d._convolution_op = functools.partial(self.masked_convolution_op, mask=mask)
        return self.conv2d.call(x)

In [3]:
mcon = MaskedConv2D(filters=2,kernel_size=[3,3])

# hack: initialize it by running some data through it
mcon((np.ones([1,4,4,3], dtype=np.float32), tf.constant([[1,1,0],[1,1,1],[0,1,1]], dtype=tf.float32)))

# set all the weights to 1 for testing
mcon.set_weights([ np.ones([3,3,3,2]) , np.zeros([2]) ])

# pass in a matrix of 1s and mask out 2 elements for each input filter
mcon((np.ones([1,4,4,3], dtype=np.float32), tf.constant([[1,1,0],[1,1,1],[0,1,1]], dtype=tf.float32)))

TypeError: __init__() got an unexpected keyword argument 'filters'

In [17]:
class MaskedConv2D(tf.keras.layers.Layer):
    def __init__(self, *args, **kwargs):
        super(MaskedConv2D, self).__init__()
        self.conv2d = tf.keras.layers.Conv2D(*args, **kwargs)
        
    def build(self, input_shape):
        self.conv2d.build(input_shape[0])
        self._convolution_op = self.conv2d._convolution_op
        
    def masked_convolution_op(self, filters, kernel, mask):
        #print(kernel.shape)
        #print(tf.reshape(mask,mask.shape+[1,1]).shape)
        result = self._convolution_op(filters, kernel)#tf.math.multiply(kernel, tf.reshape(mask, mask.shape + (1,1) )))
        #print(result.shape)
        return result
    
    def call(self, inputs):
        x = inputs
        mask=tf.constant([[1,1,0],[1,1,1],[0,1,1]], dtype=tf.float32)
        self.conv2d._convolution_op = functools.partial(self.masked_convolution_op, mask=mask)
        return self.conv2d.call(x)

In [18]:
mcon = MaskedConv2D(filters=2,kernel_size=[3,3])

# hack: initialize it by running some data through it
mcon((np.ones([1,4,4,3], dtype=np.float32) ))

# set all the weights to 1 for testing
mcon.set_weights([ np.ones([3,3,3,2]) , np.zeros([2]) ])

# pass in a matrix of 1s and mask out 2 elements for each input filter
mcon((np.ones([1,4,4,3], dtype=np.float32)))

ValueError: not enough values to unpack (expected 2, got 1)